# Entrenamiento en la nube (Google Colab) — Red neuronal para la variación anual del PIB

**Universidad Manuela Beltrán · Inteligencia Artificial**

Plataforma cloud: **Google Cloud — Google Colab** (máquina virtual de Google, sin instalar nada en el PC).

- **Datos:** base del docente, hoja *Cuadro 2 Var. % Anual* (DANE), 11 años (2001–2011).
- **Entradas (8):** x1 importaciones, x2 consumo total, x3 consumo de hogares, x4 consumo del Gobierno, x5 formación bruta de capital, x6 formación bruta de capital fijo, x7 demanda final interna, x8 exportaciones.
- **Salida (1):** variación porcentual anual del PIB (no valor en pesos).
- **Red:** perceptrón multicapa (MLP) **8 → 4 → 1**, activación `tanh`, optimizador L-BFGS.

## 1. Instalar las versiones exactas de librerías
Deben coincidir con `backend/requirements.txt`; si no, Render no podrá cargar el modelo.

**Después de ejecutar esta celda: menú *Entorno de ejecución → Reiniciar sesión*** y continúa desde la celda 2.

In [ ]:
%pip install -q numpy==2.2.6 scipy==1.15.3 scikit-learn==1.7.2 joblib==1.5.2 pandas==2.2.3 matplotlib==3.9.4

## 2. Verificar que estamos en la nube

In [ ]:
import sys, platform, os
import sklearn, numpy, joblib
print('Python      :', sys.version.split()[0])
print('Sistema     :', platform.platform())
print('Máquina     :', platform.node(), '| CPUs:', os.cpu_count())
print('scikit-learn:', sklearn.__version__, '| numpy:', numpy.__version__, '| joblib:', joblib.__version__)
assert sklearn.__version__ == '1.7.2', 'Versión distinta: reinicia la sesión después de la celda 1'
print('Entorno listo en Google Colab.')

## 3. Subir la base de conocimiento
Selecciona `backend/datos/pib_2001_2011.csv` (extracción fiel de los 11 registros del `.xls` del docente).

In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'pib_2001_2011.csv' in uploaded, 'Debes subir el archivo pib_2001_2011.csv'

## 4. Explorar la base de conocimiento

In [ ]:
import pandas as pd
df = pd.read_csv('/content/pib_2001_2011.csv')
print('Forma de la matriz:', df.shape, '→ 11 años × (año + 8 entradas + PIB)')
df.round(2)

## 5. Entrenar y evaluar la red neuronal
Validación cronológica: **2001–2008 entrenamiento**, **2009–2011 prueba**. Se compara con un predictor simple (media histórica). Luego se entrena el **modelo final con los 11 años** y se exporta.

In [ ]:
"""Entrenamiento reproducible del perceptrón multicapa (red neuronal) para PIB.
Uso: python train.py
El conjunto solo contiene 11 años (2001-2011); uso didáctico exclusivamente.
"""
from pathlib import Path
import csv
import json
import numpy as np
import joblib
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path('/content')  # carpeta de trabajo de Google Colab
CSV_PATH = ROOT / 'pib_2001_2011.csv'
FEATURES = ['importaciones', 'consumo_total', 'consumo_hogares', 'consumo_gobierno',
            'formacion_capital', 'formacion_capital_fijo', 'demanda_interna', 'exportaciones']
LABELS = ['Importaciones totales', 'Consumo total', 'Consumo de hogares',
          'Consumo final del Gobierno', 'Formación bruta de capital',
          'Formación bruta de capital fijo', 'Demanda final interna', 'Exportaciones totales']


def load_data():
    with CSV_PATH.open(encoding='utf-8', newline='') as f:
        data = list(csv.DictReader(f))
    x = np.array([[float(r[k]) for k in FEATURES] for r in data], dtype=float)
    y = np.array([float(r['pib']) for r in data], dtype=float)
    years = [int(r['anio']) for r in data]
    return x, y, years


def build_model():
    # MLP = red neuronal de tipo perceptrón multicapa; estandarización indispensable
    # para entrenar con variables expresadas en porcentajes de distintas escalas.
    net = Pipeline([
        ('escalador_entradas', StandardScaler()),
        ('red_neuronal', MLPRegressor(hidden_layer_sizes=(4,), activation='tanh',
                                    solver='lbfgs', alpha=2.0, max_iter=2500,
                                    random_state=42))
    ])
    return TransformedTargetRegressor(regressor=net, transformer=StandardScaler())


def main():
    x, y, years = load_data()
    if x.shape != (11, 8) or len(y) != 11:
        raise ValueError(f'Base inesperada: {x.shape}. Se esperan 11 filas y 8 indicadores.')
    # Validación cronológica: entrenar 2001–2008; probar 2009–2011.
    cut = 8
    test_model = build_model()
    test_model.fit(x[:cut], y[:cut])
    pred = test_model.predict(x[cut:])
    baseline = np.full(len(y[cut:]), y[:cut].mean(), dtype=float)
    report = {
        'tipo': 'Red neuronal MLP de regresión (8 entradas, 4 neuronas ocultas, 1 salida)',
        'objetivo': 'Variación porcentual anual del PIB (no valor monetario)',
        'fuente': 'Archivo de la asignatura; hoja Cuadro 2 Var. % Anual; Fuente: DANE',
        'n_ejemplos': len(y),
        'n_variables': x.shape[1],
        'entrenamiento_cronologico': [years[0], years[cut-1]],
        'prueba_cronologica': [years[cut], years[-1]],
        'mae_red_puntos_porcentuales': round(float(mean_absolute_error(y[cut:], pred)), 4),
        'rmse_red_puntos_porcentuales': round(float(root_mean_squared_error(y[cut:], pred)), 4),
        'mae_baseline_media_puntos_porcentuales': round(float(mean_absolute_error(y[cut:], baseline)), 4),
        'casos_prueba': [
            {'anio': int(years[cut+i]), 'real': round(float(y[cut+i]), 4),
             'predicho': round(float(pred[i]), 4)} for i in range(len(pred))
        ],
        'limitaciones': 'Muestra extremadamente pequeña (11 registros). No usar para proyecciones económicas reales.'
    }
    # Modelo de demostración: tras evaluar, entrenar con toda la base original.
    final_model = build_model()
    final_model.fit(x, y)
    joblib.dump({'modelo': final_model, 'caracteristicas': FEATURES}, ROOT / 'modelo_pib.joblib')
    with (ROOT / 'reporte_entrenamiento.json').open('w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    config = {
        'nombre': 'Predicción académica de la variación anual del PIB',
        'unidad': '% de variación anual',
        'n_registros': len(y),
        'periodo': [years[0], years[-1]],
        'campos': [
            {'id': k, 'nombre': lab, 'min': round(float(x[:,i].min()), 3),
             'max': round(float(x[:,i].max()), 3)}
            for i,(k,lab) in enumerate(zip(FEATURES, LABELS))
        ],
        'historico': [
            {'anio': int(years[i]), 'pib': round(float(y[i]), 4),
             'entradas': [round(float(v), 5) for v in x[i]]}
            for i in range(len(y))
        ]
    }
    with (ROOT / 'config.json').open('w', encoding='utf-8') as f:
        json.dump(config, f, ensure_ascii=False, indent=2)
    print('Entrenamiento completado. Modelo: 8 → 4 → 1')
    print('Evaluación cronológica 2009–2011:')
    print('MAE de la red:', report['mae_red_puntos_porcentuales'], 'puntos porcentuales')
    print('MAE baseline:', report['mae_baseline_media_puntos_porcentuales'], 'puntos porcentuales')
    for item in report['casos_prueba']:
        print(f"{item['anio']}: real {item['real']}%, predicho {item['predicho']}%")
    print('Modelo y reporte guardados en:', ROOT)


main()


## 6. Revisar resultados y arquitectura

In [ ]:
import json
report = json.loads(Path('/content/reporte_entrenamiento.json').read_text(encoding='utf-8'))
print(json.dumps(report, ensure_ascii=False, indent=2))

modelo = joblib.load('/content/modelo_pib.joblib')['modelo']
red = modelo.regressor_.named_steps['red_neuronal']
print('\nCapas (pesos):', [w.shape for w in red.coefs_], '→ 8 entradas, 4 ocultas, 1 salida')
print('Iteraciones de entrenamiento:', red.n_iter_)

In [ ]:
import matplotlib.pyplot as plt
casos = report['casos_prueba']
anios = [c['anio'] for c in casos]
plt.figure(figsize=(6,3.5))
plt.plot(df['anio'], df['pib'], 'o-', label='PIB real (var. %)')
plt.plot(anios, [c['predicho'] for c in casos], 's--', label='Predicción red (prueba)')
plt.axvspan(2008.5, 2011.5, alpha=.1)
plt.xlabel('Año'); plt.ylabel('% variación anual'); plt.legend(); plt.title('Real vs. predicho — conjunto de prueba')
plt.tight_layout(); plt.show()

## 7. Descargar el modelo entrenado
Reemplaza los tres archivos del mismo nombre en la carpeta `backend/` del proyecto y súbelos a GitHub.

In [ ]:
files.download('/content/modelo_pib.joblib')
files.download('/content/reporte_entrenamiento.json')
files.download('/content/config.json')